# MountainCar with every RL family: a step-by-step notebook (Kaggle / Colab)

This notebook uses the [`mountaincar-rl`](https://github.com/benisalla/mountaincar-rl) package to train, explain and compare on the
same sparse-reward task:

| family | methods (`algo`) |
|---|---|
| tabular, discretized state | `q-learning`, `sarsa` |
| linear function approximation | `tile-coding` (semi-gradient SARSA or Q-learning) |
| value-based deep RL | `dqn`, `double-dqn`, `dueling-dqn` (+ prioritized replay, n-step) |
| on-policy policy gradient | `reinforce` (with baseline), `a2c`, `ppo` |
| off-policy actor-critic (continuous) | `ddpg`, `td3`, `sac` |
| black-box search | `cem` (cross-entropy method) |

plus the exploration aids: potential-based **reward shaping** (energy or height), **count-based**
and **RND** intrinsic bonuses, and epsilon schedules.

Everything runs on a CPU. A GPU does not help much here (the networks have a few thousand weights
and the environment is the bottleneck), so pick a CPU session on Kaggle or Colab. Set `FULL = True`
in section 8 for the longer, multi-seed comparison.

**Sections:** 1 Install · 2 The problem: sparse reward · 3 Three state representations ·
4 Tabular · 5 Tile coding · 6 DQN family · 7 Policy gradient and CEM · 8 Continuous control ·
9 Exploration aids · 10 Full comparison · 11 Watch the car

## 1. Install

The repository is private: on Kaggle add a secret `GITHUB_TOKEN` (*Add-ons -> Secrets*), on Colab
put it in the *Secrets* panel, or paste a token below. If the repo is public, no token is needed.
`gymnasium[classic-control]` brings pygame, used headless (`SDL_VIDEODRIVER=dummy`) to draw the car.

In [ ]:
import os
import subprocess
import sys

os.environ["SDL_VIDEODRIVER"] = "dummy"
TOKEN = os.environ.get("GITHUB_TOKEN", "")
try:  # Kaggle secrets
    from kaggle_secrets import UserSecretsClient
    TOKEN = TOKEN or UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
try:  # Colab secrets
    from google.colab import userdata
    TOKEN = TOKEN or userdata.get("GITHUB_TOKEN")
except Exception:
    pass
url = "https://github.com/benisalla/mountaincar-rl".replace("https://", f"https://{TOKEN}@" if TOKEN else "https://")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"git+{url}.git"])

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, display

import mountaincar_rl
from mountaincar_rl.config import TrainConfig, recommended_config
from mountaincar_rl.envs import energy, make_env, state_grid
from mountaincar_rl.metrics import markdown_table, summary_row
from mountaincar_rl.train import train
from mountaincar_rl.viz import EXTENT, plot_agent_maps, plot_curves, plot_track

torch.set_num_threads(1)
print("torch", torch.__version__)

## 2. The problem: a sparse reward and a weak engine

The state is `(position, velocity)`. The track is `y = sin(3x)`; the car starts at rest near the
bottom and the flag is at `x = 0.5`. The engine force (0.001) is smaller than gravity on the slope
(up to 0.0025), so driving right fails: the car has to swing back and forth to build energy.

The reward is **-1 every step** until the flag or 200 steps. Every failed episode returns exactly
-200, whatever the agent did. A method only learns something useful *after* it reached the flag
at least once by itself. That is the exploration problem, and the whole notebook is about how each
method gets (or does not get) that first success.

Two policies below: random actions (fails), and the one-line rule "push in the direction you are
moving" (succeeds). The task is easy to *act* in, hard to *discover*.

In [ ]:
def rollout(env, policy, seed):
    obs, _ = env.reset(seed=seed)
    total, done = 0.0, False
    while not done:
        obs, r, term, trunc, _ = env.step(policy(obs))
        total, done = total + r, term or trunc
    return total, term

env = make_env()
rng = np.random.default_rng(0)
rand = [rollout(env, lambda s: int(rng.integers(3)), i) for i in range(20)]
pump = [rollout(env, lambda s: 2 if s[1] >= 0 else 0, i) for i in range(20)]
print("random policy:          success", np.mean([r[1] for r in rand]), "return", np.mean([r[0] for r in rand]))
print("push-with-velocity rule: success", np.mean([r[1] for r in pump]), "return", np.mean([r[0] for r in pump]))
plot_track(None); plt.show()

## 3. Three ways to represent the state

The state is continuous; a value function or a policy needs a representation of it:

1. **Discretization**: cut the box into a `bins x bins` grid. Simple and exact per cell, but
   *no generalization*: what is learnt in one cell says nothing about its neighbour.
2. **Tile coding**: several grids, each shifted by a fraction of a cell. A state activates one
   tile per grid; `Q(s,a)` is the sum of the active tiles' weights. Neighbouring states share
   most tiles, so learning spreads locally, and the overlaps keep a fine resolution. Linear,
   fast, and with guarantees.
3. **Neural network**: an MLP on normalised `(x, v)`. Generalizes globally and smoothly, but in
   an uncontrolled way, and needs replay buffers and target networks to be stable.

The tile coder in a few lines (`active` returns one feature index per tiling):


```python
# src/mountaincar_rl/algorithms/tile_coding.py
class TileCoder:
    """Uniformly offset grid tilings over the 2-D box; `active(s)` returns `tilings` feature indices."""

    def __init__(self, tilings: int = 8, tiles: int = 8, low=OBS_LOW, high=OBS_HIGH):
        self.tilings, self.tiles = tilings, tiles
        self.low, self.high = np.asarray(low, np.float64), np.asarray(high, np.float64)
        # each tiling covers (tiles + 1) cells so that the shifted grid still covers the whole box
        self.per_tiling = (tiles + 1) ** 2
        self.n_features = tilings * self.per_tiling
        # asymmetric offsets (1, 3) * k / tilings, as recommended in Sutton & Barto
        self.offsets = np.array([[(1 * k) % tilings, (3 * k) % tilings] for k in range(tilings)]) / tilings

    def active(self, obs) -> np.ndarray:
        scaled = (np.asarray(obs, np.float64) - self.low) / (self.high - self.low) * self.tiles
        coords = np.floor(scaled[None, :] + self.offsets).astype(int)
        coords = np.clip(coords, 0, self.tiles)
        return np.arange(self.tilings) * self.per_tiling + coords[:, 0] * (self.tiles + 1) + coords[:, 1]
```

In [ ]:
from mountaincar_rl.algorithms import TileCoder
tc = TileCoder(tilings=8, tiles=8)
a, b, c = tc.active([-0.5, 0.0]), tc.active([-0.49, 0.001]), tc.active([0.4, 0.06])
print("active tiles of s:", a)
print("shared with a close state:", len(set(a) & set(b)), "of 8 | with a far state:", len(set(a) & set(c)))

## 4. Tabular Q-learning and SARSA

One update per step, on the table entry of the current cell:

- **Q-learning** (off-policy): `Q(s,a) += alpha (r + gamma max_b Q(s',b) - Q(s,a))`
- **SARSA** (on-policy): `Q(s,a) += alpha (r + gamma Q(s',a') - Q(s,a))`, where `a'` is the action
  that will really be taken next (the agent picks it inside the update and stores it).

**Why it can work without any exploration trick:** the table starts at 0 and every true return
is negative. Tried actions drop below untried ones, so even a fully greedy agent (epsilon = 0)
keeps trying new actions in new cells. This is *optimistic initialization*. Its weakness: every
one of the 16 x 16 x 3 entries has to be learnt on its own, so it needs thousands of episodes.


```python
# src/mountaincar_rl/algorithms/tabular.py
def observe(self, obs, action, reward, next_obs, terminated, truncated) -> None:
    s, s2 = discretize(obs, self.cfg.bins), discretize(next_obs, self.cfg.bins)
    if terminated:
        target = reward                                   # the flag: no future
    elif self.method == "q-learning":
        target = reward + self.cfg.gamma * self.q[s2].max()
    else:                                                 # SARSA: bootstrap on the action we will take
        a2 = self._eps_greedy(s2)
        target = reward + self.cfg.gamma * self.q[s2 + (a2,)]
        if not truncated:
            self._next_action = a2
    self.q[s + (action,)] += self.cfg.alpha * (target - self.q[s + (action,)])
```

In [ ]:
histories = {}
for algo in ("q-learning", "sarsa"):
    cfg = recommended_config(algo, episodes=3000, log_every=500, output_dir=f"runs/nb/{algo}")
    agent, h, visits, ev = train(cfg)
    histories[algo] = h
    plot_agent_maps(agent, visits, None, title=algo); plt.show()
plot_curves(histories, None, title="tabular"); plt.show()

Reading the maps: **left** where the car went during training (log counts), **middle** the
learned `max_a Q(s,a)` (higher = closer to the flag in steps), **right** the greedy action. A
solved MountainCar shows a spiral: push right when moving right, left when moving left.

## 5. Tile coding: semi-gradient SARSA (Sutton & Barto, Example 10.1)

`Q(s,a) = w_a . x(s)` with `x(s)` the binary tile features. The gradient of a linear function
w.r.t. its weights is the feature vector, so the update moves only the 8 active weights:
`w[a, active(s)] += alpha / 8 * (target - Q(s,a))`. It is *semi*-gradient because the target
also depends on `w` and is treated as a constant. Optimistic zeros again do the exploring, and
generalization makes a single success spread to the neighbourhood: about a hundred episodes.


```python
# src/mountaincar_rl/algorithms/tile_coding.py
def observe(self, obs, action, reward, next_obs, terminated, truncated) -> None:
    feats = self.coder.active(obs)
    q_sa = self.w[action, feats].sum()
    if terminated:
        target = reward
    elif self.cfg.td_target == "q":
        target = reward + self.cfg.gamma * self.q_values(next_obs).max()
    else:
        a2 = self._eps_greedy(next_obs)
        target = reward + self.cfg.gamma * self.w[a2, self.coder.active(next_obs)].sum()
        if not truncated:
            self._next_action = a2
    # gradient of w_a . x(s) w.r.t. w_a is x(s): only the active tiles move
    self.w[action, feats] += self.step_size * (target - q_sa)
```

In [ ]:
cfg = recommended_config("tile-coding", episodes=300, output_dir="runs/nb/tile")
tile_agent, h, visits, ev = train(cfg)
print("first success at episode", h.first_success, "| greedy eval", ev)
plot_agent_maps(tile_agent, visits, None, title="tile coding"); plt.show()

## 6. DQN, Double DQN, Dueling DQN (+ prioritized replay, n-step)

A neural Q-network trained by regression on a replay buffer:

- **DQN**: `y = r + gamma (1 - done) max_a' Q_target(s', a')`, Huber loss, target net copied every 500 steps;
- **Double DQN**: the online net *chooses* `a'`, the target net *scores* it, which removes the
  max-operator's over-estimation;
- **Dueling**: `Q = V(s) + A(s,a) - mean A`, so `V` learns from every transition;
- **prioritized replay**: replay transitions with large TD error more often (sum tree), corrected
  by importance weights; **n-step**: `y = r_t + ... + gamma^(n-1) r_t+n-1 + gamma^n max Q(s_t+n)`.

`done` is 1 only at the flag; a time-out still bootstraps (the clock stopped, the state is fine).

**The lesson:** without help this usually fails. The network is not optimistic and epsilon-greedy
dithering (random actions each step) almost never pumps enough energy in 200 steps. Until the
flag is in the buffer, every target says "-1 forever". Reward shaping fixes it (section 9).


```python
# src/mountaincar_rl/algorithms/dqn.py
def update(self) -> float:
    if isinstance(self.buffer, PrioritizedReplayBuffer):
        self.buffer.beta = 0.4 + 0.6 * self.progress          # anneal importance correction to 1
    b = self.buffer.sample(self.cfg.batch_size)
    obs = torch.as_tensor(b["obs"])
    actions = torch.as_tensor(b["actions"][:, 0]).long()
    rewards, dones = torch.as_tensor(b["rewards"]), torch.as_tensor(b["dones"])
    next_obs, discounts = torch.as_tensor(b["next_obs"]), torch.as_tensor(b["discounts"])
    with torch.no_grad():
        if self.double:
            best = self.q(next_obs).argmax(dim=1, keepdim=True)
            next_q = self.q_target(next_obs).gather(1, best).squeeze(1)
        else:
            next_q = self.q_target(next_obs).max(dim=1).values
        target = rewards + discounts * (1.0 - dones) * next_q
    q_sa = self.q(obs).gather(1, actions[:, None]).squeeze(1)
    td = q_sa - target
    loss = (torch.as_tensor(b["weights"]) * F.smooth_l1_loss(q_sa, target, reduction="none")).mean()
    self.opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
    self.opt.step()
    self.buffer.update_priorities(b["idx"], td.detach().numpy())
    return float(loss.detach())
```

In [ ]:
histories = {}
for shaping in ("none", "energy"):
    cfg = recommended_config("double-dqn", shaping=shaping, episodes=300, output_dir=f"runs/nb/ddqn_{shaping}")
    agent, h, visits, ev = train(cfg)
    histories[f"double-dqn shaping={shaping}"] = h
    plot_agent_maps(agent, visits, None, title=f"double DQN, shaping={shaping}"); plt.show()
plot_curves(histories, None, title="double DQN"); plt.show()

## 7. Policy gradient (REINFORCE, A2C, PPO) and the cross-entropy method

All three gradient methods push up `log pi(a|s)` in proportion to an advantage estimate:

- **REINFORCE with baseline**: after each episode, `A_t = G_t - V(s_t)` (Monte Carlo return minus a
  learned baseline, which only reduces variance);
- **A2C**: every 16 steps, `A_t` = n-step return bootstrapped with the critic, minus `V(s_t)`;
- **PPO**: every 2048 steps, GAE advantages, then 10 epochs of minibatch updates on the clipped
  ratio objective `min(rho A, clip(rho, 0.8, 1.2) A)`; the clip keeps each update small.

With a raw -1 reward every episode returns -200, the advantages are noise, and nothing is learnt:
they need shaping. **CEM** needs no gradient: it samples 20 linear policies, keeps the best 4,
and refits a Gaussian to them. It works because a linear policy can solve the task.


```python
# src/mountaincar_rl/algorithms/policy_gradient.py
def _advantages(self, obs, lam: float):
    """GAE: delta_t = r_t + gamma (1 - term_t) V(s'_t) - V(s_t); A_t = delta_t + gamma lam (1 - end_t) A_t+1."""
    b = self.buf
    with torch.no_grad():
        v = self.net.value(obs).numpy()
        v_next = self.net.value(torch.as_tensor(np.array(b["next_obs"]))).numpy()
    rewards, term, ends = (np.array(b[k], np.float32) for k in ("rewards", "terminated", "ends"))
    adv, last = np.zeros_like(rewards), 0.0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + self.cfg.gamma * (1.0 - term[t]) * v_next[t] - v[t]
        last = delta + self.cfg.gamma * lam * (1.0 - ends[t]) * last
        adv[t] = last
    return torch.as_tensor(adv), torch.as_tensor(adv + v)

def _update_actor_critic(self, lam: float, epochs: int, minibatch: int | None) -> None:
    obs, actions = self._tensors()
    adv, returns = self._advantages(obs, lam)
    old_logp = torch.as_tensor(np.array(self.buf["logps"], np.float32))
    n = len(adv)
    for _ in range(epochs):
        order = torch.randperm(n) if minibatch else torch.arange(n)
        for start in range(0, n, minibatch or n):
            idx = order[start:start + (minibatch or n)]
            a = adv[idx]
            if minibatch and len(idx) > 1:
                a = (a - a.mean()) / (a.std() + 1e-8)
            logp, ent = self._log_prob_entropy(obs[idx], actions[idx])
            if self.cfg.algo == "ppo":
                ratio = (logp - old_logp[idx]).exp()
                clipped = ratio.clamp(1 - self.cfg.ppo_clip, 1 + self.cfg.ppo_clip)
                policy_loss = -torch.min(ratio * a, clipped * a).mean()
            else:
                policy_loss = -(logp * a).mean()
            value_loss = ((self.net.value(obs[idx]) - returns[idx]) ** 2).mean()
            self._step(policy_loss + 0.5 * value_loss - self.cfg.entropy_coef * ent.mean())
    self._clear()
```

```python
# src/mountaincar_rl/algorithms/cem.py
def end_episode(self) -> None:
    self.fitness.append(self._episode_return)
    if len(self.fitness) == self.cfg.cem_population:
        elite = self.population[np.argsort(self.fitness)[-self.n_elite:]]
        extra = max(0.0, 0.5 * (1.0 - 2.0 * self.progress))   # decaying extra noise, avoids early collapse
        self.mean, self.sigma = elite.mean(axis=0), elite.std(axis=0) + extra
        self._new_population()
```

In [ ]:
rows, histories = [], {}
for algo, extra in [("reinforce", {"shaping": "energy"}), ("a2c", {"shaping": "energy"}),
                    ("ppo", {"shaping": "energy"}), ("ppo", {}), ("cem", {})]:
    name = algo + (" +shaping" if extra else "")
    cfg = recommended_config(algo, output_dir=f"runs/nb/{name.replace(' +', '_')}", **extra)
    agent, h, visits, ev = train(cfg)
    histories[name] = h
    rows.append(summary_row(name, h, ev))
plot_curves(histories, None, title="policy gradient and CEM"); plt.show()
print(markdown_table(rows))

## 8. Continuous control: DDPG, TD3, SAC, PPO on MountainCarContinuous-v0

The force is a real number in [-1, 1], the reward is `-0.1 a^2` per step and `+100` at the flag,
for up to 999 steps. Before the first success, the only thing the critic can learn is that
pushing costs reward, so the policy drifts to **doing nothing**: a famous local optimum.

- **DDPG**: deterministic actor `mu(s)`, critic `Q(s,a)`, actor follows `dQ/da`; Gaussian action noise;
- **TD3**: two critics and the min of the two in the target, noisy target actions, delayed actor;
- **SAC**: a stochastic tanh-Gaussian actor maximising reward + `alpha` x entropy, with `alpha`
  tuned automatically; the entropy keeps it exploring longer.

The first 5 000 actions are uniform random (warm-up), which is where the first success usually
comes from.


```python
# src/mountaincar_rl/algorithms/actor_critic.py
def update(self) -> None:
    c = self.cfg
    b = self.buffer.sample(c.batch_size)
    s, a = torch.as_tensor(b["obs"]), torch.as_tensor(b["actions"])
    r, d, s2 = torch.as_tensor(b["rewards"]), torch.as_tensor(b["dones"]), torch.as_tensor(b["next_obs"])
    with torch.no_grad():
        if c.algo == "sac":
            a2, logp2 = self.actor(s2)
            q2 = torch.min(*[q(s2, a2) for q in self.critics_target]) - self.log_alpha.exp() * logp2
        elif c.algo == "td3":
            noise = (0.2 * torch.randn_like(a)).clamp(-0.5, 0.5)
            a2 = (self.actor_target(s2) + noise).clamp(-1, 1)
            q2 = torch.min(*[q(s2, a2) for q in self.critics_target])
        else:
            q2 = self.critics_target[0](s2, self.actor_target(s2))
        y = r + c.gamma * (1 - d) * q2
    critic_loss = sum(F.mse_loss(q(s, a), y) for q in self.critics)
    self.critic_opt.zero_grad()
    critic_loss.backward()
    self.critic_opt.step()

    if c.algo == "td3" and self.steps % c.policy_delay != 0:
        return
    if c.algo == "sac":
        a_new, logp = self.actor(s)
        q_new = torch.min(*[q(s, a_new) for q in self.critics])
        actor_loss = (self.log_alpha.exp().detach() * logp - q_new).mean()
    else:
        actor_loss = -self.critics[0](s, self.actor(s)).mean()
    self.actor_opt.zero_grad()
    actor_loss.backward()
    self.actor_opt.step()
    if c.algo == "sac":   # temperature: raise alpha when entropy < target, lower it otherwise
        alpha_loss = -(self.log_alpha * (logp.detach() + self.target_entropy)).mean()
        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()
    self._soft_update(self.critics, self.critics_target)
    self._soft_update(self.actor, self.actor_target)
```

In [ ]:
rows = []
for algo in ("sac", "td3", "ddpg", "ppo", "cem"):
    for shaping in ("none", "energy"):
        name = f"{algo} shaping={shaping}"
        cfg = recommended_config(algo, continuous=True, shaping=shaping, eval_episodes=10, log_every=10,
                                 output_dir=f"runs/nb/cont_{algo}_{shaping}")
        agent, h, visits, ev = train(cfg)
        rows.append(summary_row(name, h, ev))
    plot_agent_maps(agent, visits, None, title=f"{algo} continuous, shaped"); plt.show()
print(markdown_table(rows))

## 9. Exploration aids

All of them are environment wrappers, so any method can use any of them:

- **Reward shaping** (`shaping="energy"` or `"height"`): add `F = gamma Phi(s') - Phi(s)`. Because
  `F` is a difference of potentials it telescopes over an episode, so the optimal policy is
  unchanged (Ng, Harada & Russell 1999), but every step that adds energy is now rewarded. It is
  *domain knowledge*: results with it are always labelled "+shaping".
- **Count-based bonus** (`intrinsic="count"`): `0.1 / sqrt(N(cell))`, large in rarely visited cells.
- **RND** (`intrinsic="rnd"`): the error of a network trained to imitate a frozen random network;
  high on states it has not been trained on, i.e. novel states.
- **Epsilon schedules** (`eps_schedule`): `const`, `linear` or `exp` decay of epsilon-greedy.


```python
# src/mountaincar_rl/envs.py
def step(self, action):
    obs, reward, terminated, truncated, info = self.env.step(action)
    info.setdefault("env_reward", float(reward))
    phi_next = 0.0 if terminated else self.potential(obs)
    shaped = reward + self.gamma * phi_next - self._phi
    self._phi = phi_next
    return obs, float(shaped), terminated, truncated, info

def __call__(self, obs: np.ndarray) -> float:
    x = torch.as_tensor(normalize_obs(obs)).unsqueeze(0)
    with torch.no_grad():
        err = float(((self.predictor(x) - self.target(x)) ** 2).mean())
    self.running = 0.99 * self.running + 0.01 * err
    self._buffer.append(x)
    if len(self._buffer) >= self.batch:
        xb = torch.cat(self._buffer)
        self._buffer = []
        loss = ((self.predictor(xb) - self.target(xb)) ** 2).mean()
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
    return err / (self.running + 1e-8)
```

In [ ]:
pos, vel, states = state_grid(80)
plt.imshow(energy(states).reshape(80, 80).T, origin="lower", extent=EXTENT, aspect="auto")
plt.colorbar(label="Phi = energy"); plt.xlabel("position"); plt.ylabel("velocity"); plt.title("shaping potential")
plt.show()
rows = []
for name, extra in [("dqn", {}), ("dqn +count", {"intrinsic": "count"}), ("dqn +rnd", {"intrinsic": "rnd"}),
                    ("dqn +height", {"shaping": "height"}), ("dqn +energy", {"shaping": "energy"})]:
    cfg = recommended_config("dqn", output_dir=f"runs/nb/expl_{name.replace(' +', '_')}", **extra)
    agent, h, visits, ev = train(cfg)
    rows.append(summary_row(name, h, ev))
print(markdown_table(rows))

## 10. Full comparison

`run_suite` trains every run of a suite with the same seed, budget rules and greedy evaluation,
and `make_figures` writes the curves, visitation small multiples, maps and a markdown table (the
README's tables come from exactly these calls, with `FULL = False`). With `FULL = True` every suite
is repeated over three seeds with twice the budget, which takes a few hours on a CPU session.

In [ ]:
from mountaincar_rl.compare import SUITES, make_figures, run_suite

FULL = False
SEEDS = [0, 1, 2] if FULL else [0]
for seed in SEEDS:
    for suite in ("tabular", "discrete-deep", "exploration", "continuous"):
        episodes = None
        if FULL:
            episodes = {"tabular": 6000, "continuous": 60}.get(suite, 600)
        rows = run_suite(suite, out=f"runs/full_seed{seed}", seed=seed, episodes=episodes, verbose=False)
        make_figures(suite, out=f"runs/full_seed{seed}", assets=f"assets_seed{seed}")
        print(f"\n### {suite}, seed {seed}\n")
        print(markdown_table(rows))
display(Image(f"assets_seed{SEEDS[0]}/discrete-deep_curves.png"))

## 11. Watch the car

The greedy tile-coding agent from section 5, rendered headless to a GIF.

In [ ]:
from mountaincar_rl.play import play
play(tile_agent, continuous=False, gif="tile_coding.gif")
display(Image("tile_coding.gif"))